# Rewrite table paths for the us-east-1 move

**Run in JupyterHub on the us-east-2 cluster** (`hub.teehr.rtiamanzi.org`). Part of RTIInternational/teehr-hub#465.

For every table in the catalog this notebook:

1. records its current snapshot ID and row count, and
2. runs Iceberg's `rewrite_table_path`, which writes copies of the table's metadata (metadata JSON, manifest
   lists, manifests) with `s3://dev-teehr-iceberg-warehouse/` replaced by `s3://teehr-iceberg-warehouse-use1/`.

The rewritten files are staged under `<table location>/_region_move/<run id>/`, inside the table's own location,
because Polaris only vends credentials for that. Nothing in the live tables changes.

Results go to `/data/region_move/<run id>/rewrite_manifest.json`. EFS replication carries that to the us-east-1
`/data`, where `04_register_tables` reads it. Next: `03_copy_staged_metadata.sh` (laptop).

**At cutover**, pause Prefect on us-east-2 and run `01_sync_buckets.sh` first, so the snapshot recorded here is the
final one. **For the dry run**, set `ONLY_TABLES` to one small table.

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

from teehr.evaluation.spark_session_utils import create_spark_session

In [ ]:
SOURCE_PREFIX = "s3://dev-teehr-iceberg-warehouse/"
TARGET_PREFIX = "s3://teehr-iceberg-warehouse-use1/"
NAMESPACES = ["teehr", "schema_evolution"]

# Test tables not carried over (decided on #465).
EXCLUDE_TABLES = {
    "teehr.nwmd_metrics_by_location_v2_test",
    "teehr.nwmd_metrics_by_location_v2_wy_string",
}

# Dry run: e.g. ["teehr.units"]. None means every table.
ONLY_TABLES = None

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT_DIR = Path("/data/region_move") / RUN_ID
RUN_ID

In [ ]:
%%time
spark = create_spark_session()
spark.sql("USE iceberg")

List the tables to migrate.

In [ ]:
tables = []
for ns in NAMESPACES:
    for row in spark.sql(f"SHOW TABLES IN iceberg.{ns}").collect():
        fq = f"{ns}.{row.tableName}"
        if fq in EXCLUDE_TABLES:
            continue
        if ONLY_TABLES is not None and fq not in ONLY_TABLES:
            continue
        tables.append(fq)

print(f"{len(tables)} tables")
tables

For each table: record the current snapshot and row count, then rewrite its paths. `count()` on an Iceberg
table without filters is answered from manifest statistics, so it's fast even for the large tables.

In [ ]:
def table_location(fq):
    rows = spark.sql(f"DESCRIBE TABLE EXTENDED iceberg.{fq}").collect()
    return next(r.data_type for r in rows if r.col_name == "Location").rstrip("/")


def current_snapshot_id(fq):
    rows = spark.sql(
        f"SELECT snapshot_id FROM iceberg.{fq}.snapshots ORDER BY committed_at DESC LIMIT 1"
    ).collect()
    return rows[0].snapshot_id if rows else None


results = []
for fq in tables:
    location = table_location(fq)
    if not location.startswith(SOURCE_PREFIX.rstrip("/")):
        raise ValueError(f"{fq} is at {location}, outside {SOURCE_PREFIX}")
    staging = f"{location}/_region_move/{RUN_ID}/"
    snapshot_id = current_snapshot_id(fq)
    row_count = spark.table(f"iceberg.{fq}").count()

    out = spark.sql(f"""
        CALL iceberg.system.rewrite_table_path(
            table => '{fq}',
            source_prefix => '{SOURCE_PREFIX}',
            target_prefix => '{TARGET_PREFIX}',
            staging_location => '{staging}'
        )
    """).collect()[0]

    results.append({
        "table": fq,
        "source_location": location,
        "target_location": location.replace(SOURCE_PREFIX.rstrip("/"), TARGET_PREFIX.rstrip("/"), 1),
        "staging_location": staging,
        "latest_version": out.latest_version,
        "file_list_location": out.file_list_location,
        "snapshot_id": snapshot_id,
        "row_count": row_count,
    })
    print(f"{fq}: snapshot {snapshot_id}, {row_count:,} rows -> {out.latest_version}")

Save the manifest for `03_copy_staged_metadata.sh` and `04_register_tables`.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
manifest = {
    "run_id": RUN_ID,
    "source_prefix": SOURCE_PREFIX,
    "target_prefix": TARGET_PREFIX,
    "tables": results,
}
manifest_path = OUT_DIR / "rewrite_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print(manifest_path)

In [ ]:
spark.stop()